# CEGMem on Colab

Runs the whole study — candidate selection, the oracle gate, the π̂ screen, the
corpus freeze, the E1–E8 grid and the analysis — against a local
`qwen2.5-coder:7b` served by Ollama on the Colab GPU, or against an OpenAI chat
model on `--backend cloud`.

`RUNBOOK.md` in the repository is the authority on what each stage does and why.
This notebook is only the Colab wrapper around it.

**Three things Colab changes, and they are not cosmetic.**

1. **The session dies.** Free-tier runtimes stop after a few hours, and idle tabs
   stop sooner; the full grid is days of wall clock. So `cache/`, `data/` and
   `logs/` live on Google Drive, and the work is cut into shards small enough to
   finish inside one session. Nothing is lost to a disconnect — one
   `RoundRecord` is written per round as it goes — but a shard that *finishes*
   leaves cleaner books than one that is interrupted.
2. **The GPU decides the context window.** Ollama picks the window from available
   VRAM and **truncates** an over-long prompt instead of refusing it, and that
   window reaches neither the response cache key nor any logged row. A T4 has
   less VRAM than a workstation, so the risk of silently getting 4096 instead of
   32768 is higher here. §11 verifies it before anything is spent; if the check
   fails, stop.
3. **Do not commit from Colab.** `data/` becomes a symlink into Drive, so
   `git status` will report the tracked `data/mutants.py` as deleted. Colab is a
   compute node; results travel back through Drive and commits are made on your
   own machine.

**A cell that blocks is a cell you cannot use.** Every long stage below is
launched into the background and returns immediately. `!tail -f` is not used
anywhere: it never returns, and a Colab tab holding a `tail -f` is a tab you
have to interrupt before you can look at anything else. `scripts/fleet.sh status`
replaces it — re-run that cell as often as you like.

## 1. Configuration

Edit this cell, then run everything below in order.


In [ ]:
# ── repository ──────────────────────────────────────────────────────────────
REPO_URL = "https://github.com/dat95201/ceg-mem.git"   # private repo: https://<TOKEN>@github.com/...
BRANCH   = "feat/data-measure-pi"
WORKDIR  = "/content/ceg-mem"

# ── where results persist across sessions ───────────────────────────────────
RUNS = "/content/drive/MyDrive/ceg-mem-runs"

# ── the ConDefects test data (several GB) ───────────────────────────────────
# Preferred: a path inside your mounted Drive. Leave as None to fall back to the
# file id below, which downloads over the network instead.
TEST_ZIP_DRIVE_PATH = None      # e.g. "/content/drive/MyDrive/ceg-mem/Test.zip"
TEST_ZIP_FILE_ID    = "https://drive.google.com/file/d/1_rP6AnDKe6mjCyyA29EdbRb4aa15x4M4/view?usp=sharing"   # bare id or a full Drive URL

# ── the protocol. These are not preferences: every one of them changes what the
#    measured quantities ARE, and a shard measured under a different value is a
#    different instrument. Keep them identical across every machine and session
#    for the life of the study. See RUNBOOK.md §1.
MODEL          = "qwen2.5-coder:7b"
CONTEXT_LENGTH = 32768
TEMPERATURE    = 1.0
OLLAMA_PORT    = 11435          # our own port, not a desktop app's

# ── how many shards a fleet cuts a range into ───────────────────────────────
# Ollama runs with OLLAMA_NUM_PARALLEL=1, so shards do NOT get parallel
# generation - their model calls queue at the server. What overlaps is the
# oracle: each candidate patch runs in a sandbox subprocess, which is CPU, off
# the GPU's critical path. So the useful count is bounded by cores, not VRAM,
# and the gain flattens fast. 4-6 is the honest range on a T4.
SHARDS = 6

# ── the second proposer (optional, §17). Leave the key empty to stay local. ──
# Only ever run on the 30-task sweep subset, never on the reported grid: pi is a
# property of the model, the corpus was banded under qwen, and `model` is in the
# cell key so the two can never pool. See RUNBOOK.md and PLAN §2.4.
CLOUD_MODEL   = "gpt-4o-mini"
CLOUD_API_KEY = ""              # sk-... ; empty means "do not run the cloud arm"
CLOUD_PRICE_IN, CLOUD_PRICE_OUT = 0.15, 0.60    # USD per Mtok, gpt-4o-mini
CLOUD_CONTEXT = 128000
CLOUD_BUDGET  = 5.0             # TOTAL for the fleet; fleet.sh divides it by SHARDS

## 2. Check the runtime has a GPU


In [ ]:
!nvidia-smi -L || echo "NO GPU — Runtime > Change runtime type > T4 GPU, then rerun"

A 7B model at Q4_K_M is about 4.7 GB and fits a T4 alongside a 32k window. On CPU
each call takes minutes rather than seconds, and the grid never finishes — so
this is a hard prerequisite, not a nicety.

## 3. Mount Drive and create the persistent directories


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
for sub in ("cache", "data", "logs"):
    os.makedirs(f"{RUNS}/{sub}", exist_ok=True)
print("persistent root:", RUNS)
!ls -la {RUNS}

## 4. Clone the repository


In [ ]:
import os; os.chdir('/content')
!test -d {WORKDIR} || git clone {REPO_URL} {WORKDIR}
import os; os.chdir(WORKDIR)
!git fetch --all --quiet && git checkout {BRANCH} && git pull --ff-only
!git log --oneline -1

## 5. Point `cache/`, `data/` and `logs/` at Drive

This is the cell that makes a multi-session run possible. `data/mutants.py` is
**source, not a result** — it is copied into Drive first so the symlink does not
hide it.

In [ ]:
import os; os.chdir(WORKDIR)
# mutants.py is tracked source; it must exist inside the Drive-backed data dir
!cp -n data/mutants.py {RUNS}/data/ 2>/dev/null; true

!rm -rf cache data logs
!ln -s {RUNS}/cache cache
!ln -s {RUNS}/data  data
!ln -s {RUNS}/logs  logs

!ls -la | grep -E ' (cache|data|logs)'
print("--- data/ must contain mutants.py ---")
!ls data/

> The response cache is thousands of small files and Drive is slow with those. If
> a stage crawls, keep `cache/` on local disk instead and copy it to Drive after
> each stage (`!rsync -a cache/ {RUNS}/cache/`). The trade is that calls bought
> since the last sync are lost when the session ends — they are re-bought, not
> corrupted.

## 6. Python dependencies


In [ ]:
import os; os.chdir(WORKDIR)
!pip install -q -r requirements.txt
!apt-get -qq install -y lsof > /dev/null
!python3 -c "import openai, dotenv, numpy, scipy, matplotlib; print('deps ok')"

## 7. Clone ConDefects (code only)


In [ ]:
import os; os.chdir(WORKDIR)
!python3 scripts/fetch_condefects.py

## 8. The contest test data

`fetch_condefects.py` can only clone the *code*. The test data ships separately,
and without it there are no inputs, so there is no oracle and nothing below runs.

Put the archive at `external/ConDefects/Test.zip` and let the script unpack it —
it also handles the mirrors that wrap the tree one level deeper (`Test/Test/…`)
and verifies the layout afterwards.

**A partial archive is fine, and is the normal case here.** The full test tree
unpacks to roughly 63 GB; an archive holding only the coding tasks in
`data/candidates.json` is a few GB. Either works, and no cell changes between
them, as long as the archive's entries are rooted at `Test/<contest>/<LETTER>/`
— unpacking into `external/ConDefects/` then produces exactly the layout
`src.adapter.test_dir_for` looks under. The name does not matter: the cell below
copies whatever you point it at to `external/ConDefects/Test.zip`, which is where
`fetch_condefects.py` expects to find it.

**One stage does need the whole tree, and it is easy to miss.** Stage 0 gates on
the shipped test data — `G1_no_expected_output` requires a coding task to ship at
least one expected output, and `G4` counts test cases — so running
`pipeline.sh candidates` against a partial archive silently produces a
*different* candidate list: thousands of faults fail G1 for want of data rather
than for anything about the fault. With a partial archive, bring
`data/candidates.json` over as an artifact instead of regenerating it. The stage
cell below checks for this and refuses rather than quietly renumbering every
shard.

Everything downstream is safe: the gate, the screen, the corpus freeze and the
grid only ever touch tasks in the candidate list, and the gate's sibling faults
are other submissions to the *same* coding task, so they share its test
directory.

**Disk.** `Test/` must stay on Colab's local disk — never symlink it into Drive,
which has nowhere near the room.

In [ ]:
import os; os.chdir(WORKDIR)
import os, re, shutil, zipfile

dst = "external/ConDefects/Test.zip"

# A Drive download that hits the virus-scan interstitial or a quota wall saves
# the HTML error page under the target name. Catch that here, loudly: left alone
# it surfaces much later as a BadZipFile, or worse as a stage that finds no data.
def check(path):
    size = os.path.getsize(path)
    if not zipfile.is_zipfile(path):
        head = open(path, "rb").read(160)
        print(f"{path} is {size/1e6:.1f} MB and is NOT a zip archive.")
        print("It is Drive's HTML error page saved under the archive's name.")
        print("first bytes:", head)
        print()
        print("Fix by either:")
        print("  - setting TEST_ZIP_DRIVE_PATH to the file inside your mounted Drive")
        print("    (no quota, no confirm token - this is the reliable path), or")
        print("  - sharing the file 'Anyone with the link' to allow an anonymous download.")
        raise SystemExit(1)
    with zipfile.ZipFile(path) as zf:
        names = zf.namelist()
    print(f"{path}: {size/2**30:.2f} GB, {len(names):,} entries, root {names[0]!r}")

if os.path.exists("external/ConDefects/Test"):
    print("Test/ already unpacked - skipping download")
elif TEST_ZIP_DRIVE_PATH and os.path.exists(TEST_ZIP_DRIVE_PATH):
    print("copying from the mounted Drive ...")
    shutil.copyfile(TEST_ZIP_DRIVE_PATH, dst)
    check(dst)
else:
    # gdown wants a bare file id. Handed a /file/d/<id>/view?usp=... URL it
    # downloads the VIEW PAGE - a few dozen KB of HTML under the archive's name,
    # which is the failure this extraction exists to prevent. Accept either form.
    m = (re.search(r"/d/([A-Za-z0-9_-]{20,})", TEST_ZIP_FILE_ID)
         or re.search(r"[?&]id=([A-Za-z0-9_-]{20,})", TEST_ZIP_FILE_ID))
    FILE_ID = m.group(1) if m else TEST_ZIP_FILE_ID.strip()
    print("downloading file id:", FILE_ID)
    !pip install -q gdown
    !gdown {FILE_ID} -O {dst}
    check(dst)

!df -h /content | tail -1

In [ ]:
!unzip external/ConDefects/Test.zip -d external/ConDefects/

### Does the archive cover what the study needs?

A partial archive is only safe if it covers every task the pipeline will walk.
This is the cheapest place to find out — the alternative is a stage failing hours
in, or worse, a screen quietly reporting `pi_hat = 0.0` for a task whose oracle
had nothing to test against.

In [ ]:
import os; os.chdir(WORKDIR)
import sys, json; sys.path.insert(0, ".")
from src.adapter import discover, test_dir_for, TEST_DIR

TASKS = discover()
task_ids  = {t.task_id for t in TASKS.values()}
with_data = {tid for tid in task_ids if test_dir_for(tid) is not None}
print(f"coding tasks in Code/      : {len(task_ids)}")
print(f"of those, with test data   : {len(with_data)}")
print(f"archive looks              : {'PARTIAL' if len(with_data) < 0.9*len(task_ids) else 'complete'}")

# If the candidate list is already here, check the archive covers all of it.
if os.path.exists("data/candidates.json"):
    names = [c["name"] for c in json.load(open("data/candidates.json"))["candidates"]]
    missing = [n for n in names if n.split("/")[0] not in with_data]
    print(f"\ncandidates                 : {len(names)}")
    print(f"without test data          : {len(missing)}  {missing[:5]}")
    print("OK - the archive covers every candidate" if not missing
          else "STOP - those candidates cannot be screened or run")
else:
    print("\nno data/candidates.json yet - see the candidates stage below")

## 9. Ollama, with the context window pinned


In [ ]:
import os; os.chdir(WORKDIR)
# The installer unpacks a zstd-compressed bundle and Colab images do not ship
# zstd, so it fails with "This version requires zstd for extraction".
!apt-get -qq install -y zstd curl > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

In [ ]:
import os; os.chdir(WORKDIR)
import subprocess, os, time, urllib.request

# subprocess.Popen, not `!ollama serve &`: a background job started from a `!`
# cell can die with the subshell that launched it.
os.makedirs("logs", exist_ok=True)
env = dict(os.environ,
           OLLAMA_HOST=f"127.0.0.1:{OLLAMA_PORT}",
           OLLAMA_CONTEXT_LENGTH=str(CONTEXT_LENGTH),
           OLLAMA_NUM_PARALLEL="1",
           OLLAMA_KEEP_ALIVE="60m")
subprocess.Popen(["ollama", "serve"], env=env,
                 stdout=open("logs/ollama.log", "a"), stderr=subprocess.STDOUT)

for _ in range(40):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{OLLAMA_PORT}/api/tags", timeout=2)
        print("server up on 127.0.0.1:%d" % OLLAMA_PORT); break
    except Exception:
        time.sleep(1)
else:
    print("server did not come up - see logs/ollama.log")

In [ ]:
!OLLAMA_HOST=127.0.0.1:{OLLAMA_PORT} ollama pull {MODEL}

## 10. `.env`

`.env` is gitignored, so it does not travel with the repository. This is the
client half of the protocol; `screen_shard.sh` and `eval_shard.sh` export their
own values over it, so a drifted `.env` cannot corrupt a shard — but the
interactive tools read it.

In [ ]:
import os; os.chdir(WORKDIR)
# REASONING_EFFORT is new on this branch: src/llm.py routes o-series models
# differently (max_completion_tokens, no temperature) and refuses the pairing of
# a chat model with a non-empty effort rather than silently splitting the cells.
# Empty is correct for qwen and for gpt-4o-mini.
env_text = f"""LLM_BASE_URL=http://127.0.0.1:{OLLAMA_PORT}/v1
LLM_API_KEY=unused
MODEL={MODEL}
TEMPERATURE={TEMPERATURE}
REASONING_EFFORT=
LLM_TIMEOUT_SEC=1800
LLM_MAX_RETRIES=5
LLM_CONTEXT_TOKENS={CONTEXT_LENGTH}
PRICE_IN_PER_MTOK=0
PRICE_OUT_PER_MTOK=0
BUDGET_USD_CAP=1.0
SANDBOX_TIMEOUT_SEC=30.0
SANDBOX_RECURSION_LIMIT=10000
CONDEFECTS_ROOT=external/ConDefects
CACHE_DIR=cache
CALLS_LOG=data/calls.jsonl
"""
open(".env", "w").write(env_text)
print(env_text)

## 11. Verify before spending anything

`serve_local.sh` loads the model, asks `/api/ps` what is **actually** being
served, and refuses on a mismatch. `context_length` must read 32768.

If it reads 4096, stop here. The prompt would be silently cropped, and it would
be cropped worst on the arms that carry the most accumulated evidence — which are
exactly the arms the experiment compares.

In [ ]:
import os; os.chdir(WORKDIR)
!bash scripts/serve_local.sh --port {OLLAMA_PORT}
!bash scripts/pipeline.sh

---

## 12. What this branch adds, and why it changes the run order

If you last ran this notebook for the screen, everything below §12 is different.
`git log` and `RUNBOOK.md` are the record; this is the short list of what shows
up as new *cells*.

| | |
|---|---|
| **a 4th arm** | `transcript` — the ChatRepair baseline. The `untyped` arm shows the proposer nothing, which a reviewer reads as a straw man; this is the condition real agents actually implement. `--exp E6-transcript`. |
| **2 more `c` levels** | `E5-c25`, `E5-c00`. Four points gave a slope but no crossover; `c*` needs to see typed fall below untyped. |
| **a redundancy audit** | `--exp E8-audit` pays the oracle on guarded rounds so their failure type is on the record. Without it every θ-based redundancy count is censored exactly where an arm guards. Sweep subset only — it spends the time E2 exists to show can be saved. |
| **a regression audit** | `--check-regression` scores an accepted patch on **both** halves of the shipped pool: the cases the faulty version fails (what the loop optimises) and the cases it passes (which nothing in the loop ever checks). Not in the cell key, so it can be switched on without invalidating finished cells. |
| **a cloud backend** | `--backend cloud` on both shard scripts. Same wire format, different base URL; o-series routing included. §17. |
| **3 post-hoc scripts** | `measure_redundancy.py`, `measure_patch_quality.py`, `measure_typing_coherence.py`. All `$0`, all read logs that already exist. |
| **`scripts/fleet.sh`** | N background shards of one stage, one manifest, one status command. This is what makes the cells below non-blocking. |

**The one ordering rule that costs real money to get wrong: run `E1` first.**
`E2`'s `untyped` cell and `E3-guard-only` build a byte-identical prompt to
`E1`'s, so they replay E1's cached draws for free. Running them first buys the
same draw twice.

### `fleet.sh` in one cell

```
bash scripts/fleet.sh screen --shards 5 --from 401 --to 450 --calls 40
bash scripts/fleet.sh eval   --exp E1 --shards 6
bash scripts/fleet.sh eval   --exp E2 --shards 6 -- --check-regression
bash scripts/fleet.sh status      # non-blocking, re-run freely
bash scripts/fleet.sh tail 3      # BLOCKS (it is tail -f). Ctrl-C to stop.
bash scripts/fleet.sh wait        # BLOCKS until drained, then unloads the model
bash scripts/fleet.sh stop
```

Everything after a bare `--` is forwarded verbatim to the shard script.

Three things it does that hand-typed `nohup` lines do not:

* **Ranges** are contiguous, non-overlapping and cover the range exactly, with
  the remainder at the *front* so no shard is left running alone at the end.
* **`--no-stop-model` on every shard.** Both shard scripts unload the weights on
  exit *by default and on purpose*, including when they are only borrowing a
  server. So the first of six shards to finish runs `ollama stop` on the model
  the other five are still calling, and they each pay a cold reload.
  `--keep-serving` does **not** prevent this — it keeps the server process, not
  the resident model. `fleet.sh wait` does the single unload at the end.
* **On `--backend cloud` it divides `BUDGET_USD_CAP`** by the shard count.
  `src.llm.spent()` reads only its own process's ledger and every shard has one,
  so six shards each honouring a $25 cap is a real ceiling of $150.

In [ ]:
import os; os.chdir(WORKDIR)
# Pull the branch, then prove fleet.sh is there and sees the frozen corpus.
# The three numbers it prints are read off data/eval_order.txt and
# data/sweep_programs.txt, not written down in the script - if they are "-" the
# corpus has not been frozen in this Drive yet.
!git fetch --all --quiet && git checkout {BRANCH} && git pull --ff-only
!git log --oneline -1
!bash scripts/eval_shard.sh -h | sed -n '/--from N --to M/,/writes it from/p'

---

# The pipeline

Run these in order. `pipeline.sh` refuses to start a stage whose input artifact
is missing, so a skipped step is caught rather than silently producing a smaller
result.

**Sizing a shard to a session.** At roughly 20 s per call, a session safely holds
**1,500–2,000 calls**. Cut smaller than feels necessary: an interrupted shard
loses nothing, but a finished one leaves cleaner books.

## Stage: candidates — no model calls, a few minutes

In [ ]:
import os; os.chdir(WORKDIR)
import sys, json, hashlib; sys.path.insert(0, ".")

# The candidate LIST reproduces on a partial tree: dedup() draws once per coding
# task over the faults that pass, and every task that survives to G6 has its test
# data here, so the RNG stream is identical. What does NOT reproduce is
# control_unfiltered (drawn over G1-only survivors, a population this tree does
# not contain) and funnel (attribution shifts to G1). Take those two from a
# full-tree run; this file's copies of them are not usable.
EXPECTED_ORDER_SHA = "ac1bd4508978fc80798487a534fdb7b43deff2a17438aa3b33f73c97062db21d"

!bash scripts/pipeline.sh candidates

d = json.load(open("data/candidates.json"))
names = [c["name"] for c in d["candidates"]]
digest = hashlib.sha256("\n".join(names).encode()).hexdigest()

print("candidates      :", d["n_candidates"])
print("distinct tasks  :", len({n.split('/')[0] for n in names}))
print("order sha256    :", digest)
print("matches full-tree run:", digest == EXPECTED_ORDER_SHA)
print()
print("funnel (NOT comparable to a full-tree run):", json.dumps(d["funnel"]))
print("control arm     :", len(d["control_unfiltered"]), "tasks - drawn over this tree only, do not report")

This decides which faults the study is ever allowed to see, and its output order
is the traversal every later index range is cut from. Run it **once** and treat
it as frozen — re-running it with different flags renumbers every shard.

## Stage: the oracle gate — no model calls, ~2 h

Launched in the background. The second cell is the progress check — re-run it as
often as you like; it returns immediately. This is the pattern used for every
long stage below.

In [ ]:
import os; os.chdir(WORKDIR)
!nohup bash scripts/pipeline.sh gate --jobs 1 > logs/gate.log 2>&1 &
print("launched; check with the next cell")

In [ ]:
import os; os.chdir(WORKDIR)
!pgrep -fa validate_oracle.py | head -2 || echo "not running"
!grep -cE '  (PASS|SKIP|INEL)  ' logs/gate.log || true
!tail -4 logs/gate.log

## Stage: the π̂ screen — a fleet, uses the model

Shards are contiguous index ranges over `data/candidates.json` and **must not
overlap** — `fleet.sh` is what guarantees that. Any contiguous range is already
balanced, so a shard is a smaller screen rather than a skewed one.

`--calls 40` is not optional in practice: π̂ lives on a grid of `1/K`, and at
`K = 10` nothing can land in `hard` = [0.02, 0.08) at all — the band the paper
predicts its largest effect in. `fleet.sh` warns if you omit it and names the
depth `data/screen_merged.json` was measured at.

In [ ]:
import os; os.chdir(WORKDIR)
FROM, TO = 1, 526
!bash scripts/fleet.sh screen --shards {SHARDS} --from {FROM} --to {TO} --calls 40 --port {OLLAMA_PORT}

In [ ]:
import os; os.chdir(WORKDIR)
!bash scripts/fleet.sh status

In [ ]:
import os; os.chdir(WORKDIR)
# Coverage is NOT proven by a drained fleet - that only says the processes
# exited. This is the audit.
!python3 scripts/consolidate_screens.py

## Stage: freeze the corpus — no model calls

In [ ]:
import os; os.chdir(WORKDIR)
!bash scripts/pipeline.sh corpus

import json, collections
d = json.load(open("data/tasks.json"))
print(d["n_selected"], "tasks;", dict(collections.Counter(t["stratum"] for t in d["tasks"])))
print("screen depth K =", d["selection"]["min_calls"])

> **Read that output before going further.** π̂ lives on a grid of `1/K`, so a
> band can only be filled if some multiple of `1/K` falls inside it. At `K = 10`,
> for example, nothing can land in `hard` = [0.02, 0.08) at all — and that is the
> band where the paper predicts its largest effect.
>
> An empty primary band is fixed by deepening the screen, **not** by proceeding.
> It has to happen now: re-freezing the corpus once episodes exist invalidates
> them, and `eval_shard.sh` refuses to cut a shard from a corpus whose digest no
> longer matches the order it was built from.

## Stage: the oracle's blind spot — no model calls

In [ ]:
import os; os.chdir(WORKDIR)
!bash scripts/pipeline.sh pool-strength --jobs 4

---

## Stage: the grid — fleets, uses the model

### First the trial, then a smoke test of every preset that has never run

The trial is three tasks, one seed, B=5, on a log that can never be merged into
reported data. It is not a smoke test of the model, it is a test of the flags and
of the server.

Then the same for the four presets this branch adds. The check that matters is
the **second** run of each: every cell must print `already complete, skipping`,
in seconds. Anything else means the resume key does not match the index — and a
multi-day grid is the wrong place to discover that. This branch put
`audit_guarded`, `transcript_window` and `reasoning_effort` into the cell key, so
this is exactly the run that would catch a mistake in that.

In [ ]:
import os; os.chdir(WORKDIR)
!bash scripts/pipeline.sh eval --exp trial --port {OLLAMA_PORT}
!python3 scripts/summarize.py --episodes-path data/episodes_trial.jsonl

In [ ]:
import os; os.chdir(WORKDIR)
for exp in ["E3-guard-only", "E3-steer-only", "E4-k3", "E5-c50",
            "E5-c25", "E5-c00", "E5-random", "E6-transcript", "E8-audit"]:
    print(f"\n===== {exp} =====")
    !bash scripts/pipeline.sh eval --exp {exp} --from 1 --to 1 --seeds 1 --budget 5 --port {OLLAMA_PORT}

In [ ]:
import os; os.chdir(WORKDIR)
# Run it again. Every line must say "already complete, skipping".
for exp in ["E5-c25", "E5-c00", "E5-random", "E6-transcript", "E8-audit"]:
    print(f"\n===== {exp} (second run) =====")
    !bash scripts/pipeline.sh eval --exp {exp} --from 1 --to 1 --seeds 1 --budget 5 --port {OLLAMA_PORT} 2>&1 | tail -3

In [ ]:
import os; os.chdir(WORKDIR)
# Clear the rehearsal. The cached completions stay and E1 replays them free.
!rm -f data/episodes_trial.jsonl data/overfit_trial.jsonl data/calls_trial.jsonl

### What each arm costs, before you start it

Run this cell before the grid and again whenever you re-freeze anything. It is
the answer to "how long will this take", derived from **your** corpus rather
than from a number somebody typed once: `E[rounds]` comes from the π̂ these 106
tasks were banded on, and the latency from your own call ledgers.

Read the `free arms` line. Three of the arms build a prompt byte-identical to
one already bought, so they cost **zero model calls** — that is what the
common-random-numbers design buys, and it is also the falsifier: those arms must
agree with E1 about `success@B` exactly, or the guard is unsound.

In [ ]:
import os; os.chdir(WORKDIR)
# ── the run plan, computed from YOUR corpus, not written down ────────────────
# Every number below is derived: E[rounds] from the pi_hat this corpus was
# actually banded on, latency from your own call ledgers. Re-freeze the corpus
# or screen deeper and this cell tells you the new answer instead of the old
# one. That is the same discipline fleet.sh applies to universe sizes.
import json, glob, statistics as st, pathlib

B = 20
tasks = json.load(open("data/tasks.json"))["tasks"]
screen = json.load(open("data/screen_merged.json"))["per_program"]
sweep = [l.strip() for l in open("data/sweep_programs.txt")
         if l.strip() and not l.startswith("#")]
NC, NS = len(tasks), len(sweep)

def e_rounds(pi):
    """E[min(first accept, B)] - a memory arm stops at its first accept, so the
    band structure, not the task count, decides what an arm costs."""
    return float(B) if pi <= 0 else (1 - (1 - pi) ** B) / pi

pi = {t["name"]: screen[t["name"]]["pi_hat"] for t in tasks}
er = {n: e_rounds(p) for n, p in pi.items()}
ER_C, ER_S = sum(er.values()), sum(er[n] for n in sweep)

secs = []
for p in glob.glob("data/calls_*.jsonl"):
    for line in open(p):
        try: d = json.loads(line)
        except Exception: continue
        if isinstance(d.get("sec"), (int, float)): secs.append(d["sec"])
LAT_MED = st.median(secs) if secs else 20.0
LAT_MEAN = st.mean(secs) if secs else 34.0

print(f"corpus {NC} tasks | sweep {NS} | latency measured on {len(secs):,} calls: "
      f"median {LAT_MED:.1f}s, mean {LAT_MEAN:.1f}s\n")
print(f"{'band':10s} {'n':>3s} {'pi mean':>8s} {'E[rounds]':>10s} {'share':>7s}")
for b in ("dead", "hard", "medium", "easy", "too_easy"):
    ns = [t["name"] for t in tasks if t["stratum"] == b]
    if not ns: continue
    s = sum(er[n] for n in ns)
    print(f"{b:10s} {len(ns):3d} {st.mean(pi[n] for n in ns):8.4f} "
          f"{st.mean(er[n] for n in ns):10.2f} {s/ER_C*100:6.1f}%")

# (exp, universe rounds, n modes, seeds, pays_for_calls)
# pays_for_calls=False means the arm builds a prompt byte-identical to one
# already bought, so every draw is a cache hit. That is the CRN design, and it
# is also the guard-soundness falsifier: those arms MUST agree with E1 about
# success@B, exactly.
PLAN = [
    ("E1",             NC * B, 1, 5, True,  "--force-full-budget: 20 rounds, no early stop"),
    ("E2 untyped",     ER_C,   1, 5, False, "prompt identical to E1 -> cache"),
    ("E2 typed",       ER_C,   1, 5, True,  "memory in the prompt"),
    ("E3-guard-only",  ER_C,   1, 3, False, "steer off -> E1's prompt -> cache"),
    ("E3-steer-only",  ER_C,   1, 3, True,  "typed prompt, guard off"),
    ("E4-k20/k8/k3",   ER_S,   3, 3, True,  "upper bound: a weaker oracle accepts sooner"),
    ("E5 x5 +random",  ER_S,   6, 3, True,  "6 levels on the sweep"),
    ("E6-transcript",  ER_C,   1, 5, True,  "LOW end; see the note below"),
    ("E8-audit",       ER_S,   2, 3, False, "same draws as E2 -> cache; oracle time only"),
]
print(f"\n{'experiment':16s} {'cells':>6s} {'rounds':>8s} {'new calls':>10s} "
      f"{'GPU-h':>7s} {'4h sess':>8s}  why")
print("-" * 96)
tot_cells = tot_rounds = tot_new = 0
for name, rounds_u, modes, seeds, pays, why in PLAN:
    n_universe = NC if rounds_u == ER_C else (NS if rounds_u == ER_S else NC)
    cells = n_universe * modes * seeds
    rounds = rounds_u * modes * seeds
    new = rounds if pays else 0
    hours = new * LAT_MEAN / 3600
    print(f"{name:16s} {cells:6d} {rounds:8.0f} {new:10.0f} {hours:7.1f} "
          f"{hours/4:8.1f}  {why}")
    tot_cells += cells; tot_rounds += rounds; tot_new += new
print("-" * 96)
print(f"{'TOTAL':16s} {tot_cells:6d} {tot_rounds:8.0f} {tot_new:10.0f} "
      f"{tot_new*LAT_MEAN/3600:7.1f} {tot_new*LAT_MEAN/3600/4:8.1f}")
lo, hi = tot_new * LAT_MED / 3600, tot_new * LAT_MEAN / 3600
print(f"\nGPU time: {lo:.0f}-{hi:.0f} hours = {lo/24:.1f}-{hi/24:.1f} days of T4.")
print(f"Free arms: {tot_rounds - tot_new:,.0f} of {tot_rounds:,.0f} rounds cost NO model call.")
print(f"\nE6-transcript is a RANGE, not a number. The low figure above assumes it")
print(f"behaves like no-memory. The earlier pilot had the guard block 16-19 of 20")
print(f"rounds, which would put it at full budget = {NC*B*5:,} calls "
      f"(+{(NC*B*5-ER_C*5)*LAT_MEAN/3600:.0f} GPU-h). Plan for the high end.")
print(f"\nOLLAMA_NUM_PARALLEL=1, so model calls QUEUE at the server: a fleet does")
print(f"not beat these hours, it keeps the GPU from idling on oracle time.")

### Where you got to, and how to cut an arm into sessions

15 arms, 3,746 cells, and a runtime that dies every few hours. Run this after
every reconnect: it reads the episode logs on Drive rather than a note, so it is
right even if the last session ended mid-shard or someone else ran a chunk.

`session_split(exp, calls_per_session)` is the part the shard flags do not
cover. `fleet.sh --shards N` splits a range across parallel shards **inside one
runtime**; this splits the arm across **runtimes**, which is the dimension that
kills a multi-day run. It sizes chunks on the same `E[rounds]` as the plan cell,
so a chunk of `dead` tasks comes out smaller than a chunk of `easy` ones — a
flat split by task count would hand one session four times the work of another.

In [ ]:
import os; os.chdir(WORKDIR)
# ── the worklist: what is done, what is left, how to cut it into sessions ────
# A 3-week run across dying Colab runtimes needs one place that answers "where
# was I". This reads the episode logs on Drive - the artifacts, not a note - so
# it is right after a disconnect, and right if someone else ran a shard.
import os, json, glob, collections, pathlib

# In order. E1 first is not a preference: E2's untyped arm and E3-guard-only
# replay E1's cached draws, so running them first buys the same draw twice.
ARMS = ["E1", "E2", "E3-guard-only", "E3-steer-only",
        "E4-k20", "E4-k8", "E4-k3",
        "E5-c90", "E5-c75", "E5-c50", "E5-c25", "E5-c00", "E5-random",
        "E8-audit", "E6-transcript"]

def cells_done(exp):
    """Distinct (task, mode, seed) with at least one round logged, for one exp."""
    seen = set()
    for p in glob.glob(f"data/episodes_eval_{exp}_*.jsonl") + ["data/episodes.jsonl"]:
        if not os.path.exists(p):
            continue
        for line in open(p):
            try: r = json.loads(line)
            except Exception: continue
            seen.add((r["task"], r["mode"], r.get("seed", 0)))
    return seen

print(f"{'#':>3s} {'--exp':16s} {'shard logs':>10s} {'cells seen':>11s}  state")
print("-" * 62)
for i, exp in enumerate(ARMS, 1):
    logs = glob.glob(f"data/episodes_eval_{exp}_*.jsonl")
    n = len(cells_done(exp)) if logs else 0
    state = "not started" if not logs else ("in progress" if n else "log present, no rounds")
    print(f"{i:3d} {exp:16s} {len(logs):10d} {n:11d}  {state}")
print("\nmerged corpus:", "data/episodes.jsonl present"
      if os.path.exists("data/episodes.jsonl") else "not merged yet")
print("Coverage is NOT what this shows. Run: bash scripts/pipeline.sh eval --merge --dry-run")


def session_split(exp, calls_per_session=1500, universe=None):
    """--from/--to chunks sized to survive one Colab session.

    A shard is not a session. fleet.sh splits a range across parallel shards
    inside ONE runtime; this splits the arm across runtimes, which is the
    dimension that actually kills a multi-day run. Sized on the SAME E[rounds]
    the plan cell computed, so a chunk of `dead` tasks is smaller than a chunk
    of `easy` ones - a flat split by task count would hand one session four
    times the work of another.
    """
    tasks_j = json.load(open("data/tasks.json"))["tasks"]
    screen = json.load(open("data/screen_merged.json"))["per_program"]
    order_file = ("data/sweep_programs.txt"
                  if exp.startswith(("E4", "E5", "E8")) else "data/eval_order.txt")
    order = [l.strip() for l in open(order_file) if l.strip() and not l.startswith("#")]
    B = 20
    full = exp == "E1"
    seeds = 5 if exp in ("E1", "E2", "E6-transcript") else 3
    modes = 2 if exp in ("E2", "E8-audit") else 1
    def cost(name):
        pi = screen[name]["pi_hat"]
        r = B if full else (B if pi <= 0 else (1 - (1 - pi) ** B) / pi)
        return r * seeds * modes
    print(f"\n{exp}: {len(order)} tasks over {order_file}, "
          f"{seeds} seed(s) x {modes} mode(s), ~{calls_per_session} calls/session")
    lo, run, n = 1, 0.0, 0
    for i, name in enumerate(order, 1):
        run += cost(name); n += 1
        if run >= calls_per_session or i == len(order):
            print(f"   --from {lo:3d} --to {i:3d}   {n:3d} tasks, ~{run:5.0f} calls")
            lo, run, n = i + 1, 0.0, 0


# Uncomment for the arm you are about to start. 1500 calls is ~8.6 h at the
# measured mean; drop it to 700 for a 4-hour free-tier session.
# session_split("E1", 1500)
# session_split("E6-transcript", 1500)

### E1 — run this arm first

`--force-full-budget` is what makes E1 an estimator rather than just a baseline:
the no-memory prompt is byte-identical across all rounds, so every round is an
independent draw of π. It is also the arm every unconditioned arm below replays
its cache from, which is why it goes first.

In [ ]:
import os; os.chdir(WORKDIR)
!bash scripts/fleet.sh eval --exp E1 --shards {SHARDS} --port {OLLAMA_PORT}

In [ ]:
import os; os.chdir(WORKDIR)
!bash scripts/fleet.sh status

### The remaining arms

Run one fleet at a time — `fleet.sh` refuses to launch a second while shards are
alive rather than interleaving two manifests. Change `EXP` and re-run.

| `--exp` | universe | cost |
|---|---|---|
| `E2` | 106 tasks | `untyped` replays E1's cache free; `typed` is new |
| `E3-guard-only` | 106 | replays E1's cache free |
| `E3-steer-only` | 106 | new calls |
| `E4-k20 k8 k3` | 30 (sweep) | new calls; read off `is_truly_correct`, not `accept` |
| `E5-c90 c75 c50 c25 c00` | 30 (sweep) | new calls |
| `E5-random` | 30 (sweep) | the c axis's **null** — classes assigned at random. `c=0.00` is not this |
| `E8-audit` | 30 (sweep) | no model calls, oracle time only |
| `E6-transcript` | 106 | **the expensive one — see below** |

**`E6-transcript` is the only arm that pays for every round.** Its prompt carries
the whole refuted transcript, so no round past the first shares a cache key with
E1. Roughly 5,000 calls ≈ **2 days of T4 wall clock**, i.e. several Colab
sessions. Either put it last, or run it on `--backend cloud` (§17) where it is a
few dollars and a few hours.

`-- --check-regression` on E2 turns `accept` into a verdict: it scores the
accepted patch on the cases the faulty version already passed, which nothing in
the loop ever looks at. Sandbox time, no model calls, not in the cell key.

In [ ]:
import os; os.chdir(WORKDIR)
EXP = "E2"
EXTRA = ["--check-regression"]        # [] for none
extra = " ".join(EXTRA)
!bash scripts/fleet.sh eval --exp {EXP} --shards {SHARDS} --port {OLLAMA_PORT} -- {extra}

In [ ]:
import os; os.chdir(WORKDIR)
!bash scripts/fleet.sh status

`fleet.sh wait` is the one blocking cell in this notebook. Run it when you want
the tab to hold until the fleet drains — it also does the single model unload
that every shard was told not to do itself.

In [ ]:
import os; os.chdir(WORKDIR)
!bash scripts/fleet.sh wait

In [ ]:
import os; os.chdir(WORKDIR)
# Only if you need the GPU back or got a range wrong. Shards resume on an
# identical re-launch: every model call already bought replays from the cache
# on Drive, and only the oracle - which is not cached - runs again.
!bash scripts/fleet.sh stop


### Merge, and audit the merge

The audit is the only thing that speaks to coverage. A fleet reporting every
shard `done` says every process exited 0 and nothing more.

In [ ]:
import os; os.chdir(WORKDIR)
!bash scripts/pipeline.sh eval --merge --dry-run     # coverage, gaps, protocol

In [ ]:
import os; os.chdir(WORKDIR)
!bash scripts/pipeline.sh eval --merge

---

## Stage: analysis — no model calls, all `$0`

In [ ]:
import os; os.chdir(WORKDIR)
!bash scripts/pipeline.sh analyse

In [ ]:
import os; os.chdir(WORKDIR)
# the sub-grids, each frozen against the list it actually ran over
!python3 scripts/freeze_results.py --experiment ablation
!python3 scripts/freeze_results.py --experiment oracle_sweep --sweep-programs-from data/sweep_programs.txt
!python3 scripts/freeze_results.py --experiment typing_sweep --sweep-programs-from data/sweep_programs.txt

### The three post-hoc scripts this branch adds

No model calls, so no dollars — but two of them are hours of sandbox, and one is
the second-biggest compute item in the whole study after the grid itself:

| | sandbox runs | wall clock |
|---|---|---|
| `--check-regression` (paid during E2, not here) | 4,200 for the F2P/P2P split, once per program, plus one full pass per accepted patch ≈ **31k** | 1.3–3.4 h |
| `measure_redundancy.py` · `measure_patch_quality.py` | none — they only read logs | minutes |
| **`measure_typing_coherence.py`** | 8,690 patches × min(60, pool size) ≈ **327k** | **14–36 h** |

Pilot the coherence script on 20 tasks first (~62k runs, ~4 h), read the `caps`
block it writes, and only then decide whether the full run earns its day.

And read `c_hat` against the two numbers now printed beside it. On its own it is
uninterpretable: a θ that shatters every patch into its own class scores a
perfect homogeneity of 1.000 **and** a random-baseline of 1.000 — lift zero. The
`competing` block scores the obvious alternatives (bucket by the name of the
refuting case, or by θ's `property` half alone). If θ does not beat both, the
lattice is not carrying its keep.

In [ ]:
import os; os.chdir(WORKDIR)
!python3 scripts/measure_redundancy.py        # #2,3,4,5,7,8,11,18,33 + pass@k (the repeated-sampling baseline)
!python3 scripts/measure_patch_quality.py     # #21 correct/plausible, #23 verbosity, #22 regression rate

In [ ]:
import os; os.chdir(WORKDIR)
!python3 scripts/measure_typing_coherence.py --limit-tasks 20   # pilot first, read the caps

In [ ]:
import os; os.chdir(WORKDIR)
!nohup python3 scripts/measure_typing_coherence.py > logs/coherence.log 2>&1 &
print("launched; tail logs/coherence.log with the cell below")

In [ ]:
import os; os.chdir(WORKDIR)
!pgrep -fa measure_typing_coherence.py | head -1 || echo "not running"
!tail -5 logs/coherence.log

In [ ]:
import os; os.chdir(WORKDIR)
# Run last, and run it again before submitting.
!python3 scripts/check_consistency.py

---

## 13. Optional: a second proposer on the sweep subset

**Do not replace qwen.** π is a property of the model: the 526 programs were
screened under `qwen2.5-coder:7b` and the corpus was banded on those numbers, so
switching the proposer invalidates the screen and forces it to be redone. Worse,
a stronger proposer shifts the π̂ distribution *right* — draining `dead` and
`hard`, which are exactly the two bands where three of the four theoretical
results are most visible. A stronger model can make the effect **harder** to see.

What is worth doing is *adding* a second proposer on the 30-task sweep subset.
It patches `DESIGN.md`'s own open item ("a single small proposer... ideally
checked against a stronger model on a subset") with a number instead of a
sentence, and it makes `#14 cost-of-pass` real money rather than a repricing.
`freeze_results.py` already refuses to mix two models in one freeze and `model`
is in the cell key, so there is no way for it to contaminate the main grid.

The cell refuses to run unless `CLOUD_API_KEY` is set in §1.

In [ ]:
import os; os.chdir(WORKDIR)
if not CLOUD_API_KEY:
    print("CLOUD_API_KEY is empty in §1 - skipping the cloud arm (this is the default)")
else:
    env = (f"LLM_API_KEY={CLOUD_API_KEY} "
           f"PRICE_IN_PER_MTOK={CLOUD_PRICE_IN} PRICE_OUT_PER_MTOK={CLOUD_PRICE_OUT} "
           f"BUDGET_USD_CAP={CLOUD_BUDGET} CONTEXT_LENGTH={CLOUD_CONTEXT}")
    # BUDGET_USD_CAP here is the TOTAL: fleet.sh divides it by --shards, because
    # src.llm.spent() only ever sees the ledger of its own process.
    print(f"{env}\n")
    !{env} bash scripts/fleet.sh eval --exp E1 --shards 4 -- --backend cloud --model {CLOUD_MODEL}

In [ ]:
import os; os.chdir(WORKDIR)
!bash scripts/fleet.sh status

---

# After a disconnect

Ollama does not survive a stopped runtime, and neither does `/content`. Drive
does, and so does everything in it. To carry on: run **§1** (config), **§3**
(mount), **§4** (clone), **§5** (links), **§6** (deps), **§8** (test data),
**§9** (Ollama), **§10** (`.env`), **§11** (verify) — then re-run the stage cell
you were on with the identical arguments.

The re-run is not wasted. Every model call already bought replays from the cache
on Drive; only the oracle, which is not cached, executes again.

The cell below is those steps collapsed into one, for a session that only needs
to resume.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os; os.chdir(WORKDIR)
!git pull --ff-only
!pip install -q -r requirements.txt && apt-get -qq install -y lsof > /dev/null

import subprocess, os, time, urllib.request
env = dict(os.environ, OLLAMA_HOST=f"127.0.0.1:{OLLAMA_PORT}",
           OLLAMA_CONTEXT_LENGTH=str(CONTEXT_LENGTH),
           OLLAMA_NUM_PARALLEL="1", OLLAMA_KEEP_ALIVE="60m")
subprocess.Popen(["ollama", "serve"], env=env,
                 stdout=open("logs/ollama.log", "a"), stderr=subprocess.STDOUT)
for _ in range(40):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{OLLAMA_PORT}/api/tags", timeout=2); break
    except Exception: time.sleep(1)

!bash scripts/serve_local.sh --port {OLLAMA_PORT}
!bash scripts/pipeline.sh

In [ ]:
import os; os.chdir(WORKDIR)
# Where the last fleet got to. The manifest survives the runtime because logs/
# is on Drive; the PROCESSES do not, so shards from a dead session read as
# KILLED. Re-launch the identical fleet - every call already bought replays free.
!bash scripts/fleet.sh status

---

# When something refuses

**`served context is 4096, not 32768`** — the GPU did not have room for the
window Ollama was asked for, or a server was already listening on the port with a
different one. Stop whatever is on the port, then re-run §9. Do not work around
it: the numbers would be wrong in a way nothing downstream can detect.

**`data/tasks.json missing` / `stage X reads Y`** — a stage was skipped.
`bash scripts/pipeline.sh` names the one to run.

**Every cell re-runs instead of skipping** — something in the experiment cell key
moved, and in practice it is `model`. Check the `model` field in the episode log
before assuming the resume logic is broken.

**`BudgetExceeded`** — local calls are priced at zero, so this cannot fire on a
healthy run. It is a tripwire: something has repointed the client at a paid
endpoint. Find out what rather than raising the cap.

**Drive quota, or `Transport endpoint is not connected`** — the mount dropped
mid-run. Re-run §3 and the stage cell; nothing is corrupted, because rounds are
appended one at a time.

`RUNBOOK.md` §9 is the full list, including the failure modes that are not
Colab-specific.

**`fleet.sh: a fleet is still running`** — one fleet at a time, by design: two
manifests interleaved is not a record of anything. `fleet.sh status`, then `stop`
or `wait`.

**A shard reads `KILLED`** — no pid and no recorded exit status, which is what a
dead Colab runtime leaves behind. Distinct from `exit N`, which is something the
shard found rather than something that happened to it. Re-launch the identical
fleet.

**Every shard reads `exit 2` immediately** — read one log (`fleet.sh tail 1`).
Almost always the server: `ollama not on PATH`, or nothing listening on
`OLLAMA_PORT` because §9 was skipped after a reconnect.

**`--backend cloud needs BUDGET_USD_CAP`** — export the TOTAL for the fleet and
let `fleet.sh` divide it. There is no safe default: the cap is per-process and
every shard has its own ledger.

**The screen ran but every band is empty except `dead`** — `--calls` was omitted,
so the shards took `screen_shard.sh`'s default depth instead of the study's.
π̂ lives on a grid of `1/K`. `fleet.sh` warns about this at launch.